# Stage 1 — Raw Page Structure Inspection


## 1. Priority Classification

In [47]:
import json
import re
from pathlib import Path
from collections import defaultdict

RAW_DIR = Path('../../data/raw_scraped_pages')
CLASSIF = Path('../../docs/url_classification.json')

# --- load pages ---
# raw files have two schemas:
#   schema A (newer spider): page_url / page_title / all_text_markdown / sections
#   schema B (older spider): source_url / text_markdown / links / raw_html
def get_url(page):
    return page.get('page_url') or page.get('source_url', '')

def get_title(page):
    return page.get('page_title', '')

pages = []
schema_counts = {'A': 0, 'B': 0}
for fpath in sorted(RAW_DIR.glob('*.json')):
    with open(fpath, encoding='utf-8') as f:
        p = json.load(f)
    pages.append(p)
    if 'page_url' in p:
        schema_counts['A'] += 1
    else:
        schema_counts['B'] += 1

print(f'Loaded {len(pages)} raw pages')
print(f'  Schema A (page_url / page_title / all_text_markdown): {schema_counts["A"]}')
print(f'  Schema B (source_url / text_markdown / raw_html):     {schema_counts["B"]}')

# --- load classification ---
with open(CLASSIF, encoding='utf-8') as f:
    classif = json.load(f)

entries      = classif['entries']
url_patterns = classif['url_patterns']
print(f'\nClassification loaded: {len(entries)} entries, {len(url_patterns)} url_patterns')

Loaded 177 raw pages
  Schema A (page_url / page_title / all_text_markdown): 176
  Schema B (source_url / text_markdown / raw_html):     1

Classification loaded: 68 entries, 19 url_patterns


In [48]:
# --- URL normalizer: raw pages may lack trailing slash ---
def norm(url: str) -> str:
    return url.rstrip('/')

# Build lookup: norm(url) -> entry
entry_lookup = {}
for e in entries:
    entry_lookup[norm(e['url'])] = e
    entry_lookup.setdefault(norm(e['canonical_url']), e)

def classify_url(url: str) -> dict:
    key = norm(url)
    if key in entry_lookup:
        return entry_lookup[key]
    for pat in url_patterns:
        if re.match(pat['regex'], url):
            return pat
    return {'priority': 'skip', 'url': url}

# --- classify every page ---
PRIORITY_ORDER = ['must', 'dsi-general', 'optional', 'alias', 'skip']
classified = defaultdict(list)   # priority -> list of dicts

for p in pages:
    url   = get_url(p)
    title = get_title(p)
    result   = classify_url(url)
    priority = result.get('priority', 'unlisted')
    classified[priority].append({'url': url, 'title': title})

print('Classification summary:')
for pri in PRIORITY_ORDER:
    print(f'  {pri:12s}: {len(classified[pri]):3d} pages')

Classification summary:
  must        :  14 pages
  dsi-general :   8 pages
  optional    :  10 pages
  alias       :   0 pages
  skip        : 145 pages


In [49]:
# --- Print each priority group in full ---
for priority in PRIORITY_ORDER:
    group = classified[priority]
    if not group:
        continue
    print(f'\n{"="*72}')
    print(f'  [{priority.upper()}]  ({len(group)} pages)')
    print(f'{"="*72}')
    for item in sorted(group, key=lambda x: x['url']):
        print(f'  {item["url"]}')


  [MUST]  (14 pages)
  https://datascience.uchicago.edu/education/masters-programs/in-person-program
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-project-archive
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/career-outcomes
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply
  https://datascience.uchicago.edu/education/masters-programs/ms-in-appl

In [50]:
# --- Gap check: sitemap must/should entries MISSING from raw ---
raw_url_norms = {norm(get_url(p)) for p in pages}

print('Sitemap must/should entries MISSING from raw_scraped_pages:\n')
missing = []
for e in entries:
    if e['priority'] in ('must', 'should'):
        if norm(e['url']) not in raw_url_norms and norm(e['canonical_url']) not in raw_url_norms:
            missing.append(e)

if missing:
    for e in missing:
        print(f'  [{e["priority"]}]  {e["url"]}')
else:
    print('  None — all must/should entries are present in raw data.')

Sitemap must/should entries MISSING from raw_scraped_pages:

  None — all must/should entries are present in raw data.


## 2. Header & Footer Classification

In [51]:
FOOTER_PATH = RAW_DIR.parent / 'raw_scraped_pages' / 'footer.json'

with open(FOOTER_PATH, encoding='utf-8') as f:
    footer = json.load(f)

footer_hrefs = [link['href'] for link in footer.get('links', []) if link.get('href')]
print(f'Footer hrefs: {len(footer_hrefs)} total')

# Compare against the 177 raw scraped pages
raw_url_norms = {norm(get_url(p)) for p in pages}

footer_classified = []
for href in footer_hrefs:
    tag = 'dsi-general' if norm(href) in raw_url_norms else 'footer-only'
    footer_classified.append({'url': href, 'tag': tag})

dsi_general_count = sum(1 for x in footer_classified if x['tag'] == 'dsi-general')
footer_only_count  = sum(1 for x in footer_classified if x['tag'] == 'footer-only')

print(f'  In raw pages  → dsi-general : {dsi_general_count}')
print(f'  Not in raw    → footer-only : {footer_only_count}')
print()
for item in sorted(footer_classified, key=lambda x: x['url']):
    print(f'  [{item["tag"]:12s}]  {item["url"]}')

Footer hrefs: 21 total
  In raw pages  → dsi-general : 12
  Not in raw    → footer-only : 9

  [footer-only ]  https://accessibility.uchicago.edu
  [footer-only ]  https://bsky.app/profile/dsi-uchicago.bsky.social
  [dsi-general ]  https://datascience.uchicago.edu
  [dsi-general ]  https://datascience.uchicago.edu/about
  [dsi-general ]  https://datascience.uchicago.edu/about/contact
  [dsi-general ]  https://datascience.uchicago.edu/about/jobs
  [dsi-general ]  https://datascience.uchicago.edu/about/leadership-staff
  [dsi-general ]  https://datascience.uchicago.edu/education
  [dsi-general ]  https://datascience.uchicago.edu/news-events/events
  [dsi-general ]  https://datascience.uchicago.edu/news-events/news
  [dsi-general ]  https://datascience.uchicago.edu/newsletter-archive
  [dsi-general ]  https://datascience.uchicago.edu/nondiscrimination-statement
  [dsi-general ]  https://datascience.uchicago.edu/outreach
  [dsi-general ]  https://datascience.uchicago.edu/research
  [footer